# Predictive Churn Analysis
## Customer Churn Probability Forecasting

**Objective:** Construct and evaluate a machine-learning classification pipeline for customer retention risk.

This notebook follows the task guideline:
1. Load the churn dataset and perform feature encoding (One-Hot Encoding + MinMax scaling).
2. Split the dataset into 80/20 train-test sets.
3. Train Logistic Regression and Random Forest classifiers.
4. Evaluate using Precision, Recall, F1-Score and ROC-AUC, including an ROC curve.
5. Export customer churn risk-score predictions.

**Dataset used:** `customer_churn_sample (4).csv`

> **Data limitation:** The supplied sample contains only 15 customers. The 80/20 test set therefore contains 3 customers. The resulting metrics are reproducible for this sample but should not be treated as production-grade model performance. A larger historical churn dataset should be used before deployment.


In [ ]:
# Install/import dependencies if needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, confusion_matrix, classification_report
)


In [ ]:
# Load dataset
# In Google Colab, upload `customer_churn_sample (4).csv` when prompted.
from google.colab import files
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Shape:", df.shape)
display(df.head())
print("\nMissing values:")
display(df.isna().sum())
print("\nChurn distribution:")
display(df["Churn"].value_counts())


## 1. Data Preparation

`CustomerID` is an identifier and is excluded from the model. `Churn` is the target. Numeric features are MinMax-scaled and categorical features are One-Hot encoded inside a single scikit-learn `ColumnTransformer`, preventing preprocessing leakage.


In [ ]:
# Separate features and target
X = df.drop(columns=["Churn", "CustomerID"])
y = (df["Churn"] == "Yes").astype(int)

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


In [ ]:
# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training churn rate:", y_train.mean())
print("Testing churn rate:", y_test.mean())


In [ ]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced"
    )
}

results = {}
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results[name] = {
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).T
display(results_df)


## 2. Model Evaluation

The required metrics are reported for the 20% holdout set. Because the sample is very small, a perfect score on the holdout set does **not** establish that the model will generalize to new customers.


In [ ]:
# Detailed evaluation and confusion matrices
for name, pipe in fitted_models.items():
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    print("=" * 60)
    print(name)
    print(classification_report(
        y_test, y_pred,
        target_names=["No Churn", "Churn"],
        zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))


In [ ]:
# ROC curves
plt.figure(figsize=(7, 5))

for name, pipe in fitted_models.items():
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, marker="o", label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Customer Churn Models")
plt.legend()
plt.grid(True)
plt.show()


## 3. Model Selection

For this sample, both models produce the same holdout metrics. Logistic Regression is selected as the final scoring model because it is simpler, faster, interpretable, and appropriate as a baseline for a small tabular dataset.


In [ ]:
# Select Logistic Regression as final scoring model
final_model = fitted_models["Logistic Regression"]

print("Selected model: Logistic Regression")
display(results_df.loc[["Logistic Regression"]])


## 4. Customer Churn Risk-Score Predictions

The final model is refit on all available rows and used to produce a probability of churn for every customer. Risk bands are:
- **Low:** < 0.33
- **Medium:** 0.33–0.66
- **High:** > 0.66


In [ ]:
# Refit final model on all available data
final_model.fit(X, y)

risk_scores = final_model.predict_proba(X)[:, 1]

risk_predictions = df[["CustomerID"]].copy()
risk_predictions["Churn_Risk_Score"] = risk_scores
risk_predictions["Predicted_Churn"] = np.where(risk_scores >= 0.50, "Yes", "No")
risk_predictions["Risk_Level"] = pd.cut(
    risk_scores,
    bins=[-0.001, 0.33, 0.66, 1.001],
    labels=["Low", "Medium", "High"]
)

display(risk_predictions)


In [ ]:
# Export predictions for submission
output_file = "customer_churn_risk_predictions.csv"
risk_predictions.to_csv(output_file, index=False)

print(f"Saved: {output_file}")

# Download in Google Colab
files.download(output_file)


## 5. Final Report / Conclusion

### Result
On the supplied 15-row sample, both Logistic Regression and Random Forest achieve the same 80/20 holdout performance:

- **Precision:** 1.00
- **Recall:** 1.00
- **F1-Score:** 1.00
- **ROC-AUC:** 1.00

### Business interpretation
The model pipeline can estimate individual customer churn probability and classify customers into risk levels. These scores can support retention prioritization.

### Important limitation
The sample has only **15 customers (8 non-churn, 7 churn)** and only **3 customers in the test set**. Therefore, the perfect test metrics are not sufficient evidence for production deployment. The correct next step is to train and validate the pipeline on a substantially larger historical dataset, preferably with cross-validation and a time-based validation strategy if churn is forecast over time.

### Submission deliverables
- Reproducible Google Colab/Jupyter Notebook
- Customer churn risk-score CSV
- This report/conclusion
